# 🎭 AvatarForcing — End-to-End RunPod RTX 5090 Notebook
One-Step Streaming Talking Avatars — run all cells top to bottom.

## 🖥️ Step 0 — GPU Check

In [ ]:
import subprocess, sys, os

def run(cmd, check=True, **kwargs):
    print(f'>>> {cmd}')
    result = subprocess.run(cmd, shell=True, text=True, **kwargs)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')
    return result

run('nvidia-smi')
run('nvcc --version')

## 🐍 Step 1 — Find or Install Conda

In [ ]:
import shutil, os, subprocess

CONDA_SEARCH_PATHS = [
    '/opt/conda/bin/conda',
    '/root/miniconda3/bin/conda',
    '/root/anaconda3/bin/conda',
    '/usr/local/conda/bin/conda',
]

CONDA_BIN = shutil.which('conda')
if CONDA_BIN is None:
    for p in CONDA_SEARCH_PATHS:
        if os.path.isfile(p):
            CONDA_BIN = p
            break

if CONDA_BIN:
    print(f'✅ Found conda at: {CONDA_BIN}')
    subprocess.run(f'{CONDA_BIN} --version', shell=True)
else:
    print('Installing Miniconda...')
    for c in [
        'curl -fsSL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh',
        'bash /tmp/miniconda.sh -b -p /root/miniconda3',
    ]:
        run(c)
    CONDA_BIN = '/root/miniconda3/bin/conda'

CONDA_DIR = os.path.dirname(os.path.dirname(CONDA_BIN))
print(f'CONDA_DIR = {CONDA_DIR}')

## 📁 Step 2 — Clone Repository

In [ ]:
REPO_DIR = '/workspace/AvatarForcing-inference-v1'

if not os.path.isdir(REPO_DIR):
    run(f'git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/AvatarForcing-inference-v1.git {REPO_DIR}')
else:
    print(f'✅ Repo already exists at {REPO_DIR}')

os.chdir(REPO_DIR)
run('ls -la')

## 🌿 Step 3 — Patch environment.yml & Create Conda Env

**Three patches applied to `environment.yml` before creation:**
1. `defaults` → `conda-forge` — fixes `CondaToSNonInteractiveError`
2. Remove `torch / torchvision / torchaudio` pip lines — `+cu121` tag only exists on the PyTorch index, not PyPI
3. Remove `flash-attn` pip line — needs torch already installed to compile
4. Remove `nvidia-*` and `cuda-*` pip lines — CUDA packages also require special index
5. Remove `triton` pip line — depends on torch

All removed packages are installed correctly in **Step 5**.

In [ ]:
# 3a — Accept Anaconda TOS
run(f'{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main', check=False)
run(f'{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r', check=False)
print('✅ TOS accepted.')

In [ ]:
import re, shutil as _sh

yml_path = f'{REPO_DIR}/environment.yml'
yml_bak  = f'{REPO_DIR}/environment.yml.bak'

# Always restore from original backup so re-running is idempotent
if os.path.exists(yml_bak):
    _sh.copy(yml_bak, yml_path)
    print('Restored original from backup.')
else:
    _sh.copy(yml_path, yml_bak)
    print(f'Backup saved: {yml_bak}')

with open(yml_path) as fh:
    content = fh.read()

# --- Patch 1: defaults → conda-forge ---
content = content.replace('  - defaults', '  - conda-forge')

# --- Patches 2-5: remove packages that need special index or pre-installed torch ---
# Pattern: match any pip list line containing these package prefixes
REMOVE_PREFIXES = [
    'flash.attn',   # needs torch to build
    'torch==',      # +cu121 not on PyPI
    'torchvision',  # +cu121 not on PyPI
    'torchaudio',   # +cu121 not on PyPI
    'torchmetrics', # depends on torch version
    'nvidia-',      # CUDA packages need torch index
    'cuda-',        # CUDA runtime packages
    'triton==',     # needs torch
    'pytorch-lightning', # depends on torch
]

removed = []
lines_out = []
for line in content.splitlines(keepends=True):
    stripped = line.strip().lstrip('-').strip()
    matched = any(stripped.lower().startswith(p.lower()) for p in REMOVE_PREFIXES)
    if matched:
        removed.append(stripped.split('==')[0])
    else:
        lines_out.append(line)

content = ''.join(lines_out)

with open(yml_path, 'w') as fh:
    fh.write(content)

print(f'✅ environment.yml patched — removed {len(removed)} problematic pip packages:')
for p in removed:
    print(f'   - {p}')

In [ ]:
# 3c — Create conda env
result = subprocess.run(f'{CONDA_BIN} env list', shell=True, text=True, capture_output=True)
print(result.stdout)

if 'avatarforcing' in result.stdout:
    print("✅ Conda env 'avatarforcing' already exists. Skipping.")
else:
    print('Creating conda env (may take 10–20 min)...')
    run(f'{CONDA_BIN} env create -f {REPO_DIR}/environment.yml')
    print('✅ Conda env created!')

In [ ]:
# Helper: run a command inside the avatarforcing env
def run_in_env(cmd, **kwargs):
    return run(f'{CONDA_BIN} run -n avatarforcing --no-capture-output {cmd}', **kwargs)

run_in_env('python --version')
run_in_env('pip --version')

## 🔧 Step 4 — Install System Dependencies (FFmpeg)

In [ ]:
run('apt-get update -qq && apt-get install -y ffmpeg libsm6 libxext6 -qq')
run('ffmpeg -version 2>&1 | head -3')

## ⚡ Step 5 — Install PyTorch (cu126) + flash-attn for RTX 5090

RTX 5090 = Blackwell / SM_100.  
We install `cu126` PyTorch wheels (from the official PyTorch index),  
then install **flash-attn via a pre-built wheel** (seconds, not hours!).

| Component | Version |
|---|---|
| CUDA | 12.6 (cu126) |
| PyTorch | 2.6.0 |
| Python | 3.11 |
| flash-attn | 2.7.4 (pre-built binary) |

In [ ]:
# 5a — Install PyTorch cu126 + torchvision + torchaudio
print('Installing PyTorch cu126 (torch 2.6.0)...')
run_in_env(
    'pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 '
    '--index-url https://download.pytorch.org/whl/cu126'
)
print('✅ PyTorch cu126 installed.')

In [ ]:
# 5b — Install pytorch-lightning and torchmetrics (compatible with torch 2.6.0)
run_in_env('pip install pytorch-lightning torchmetrics -q')
print('✅ pytorch-lightning + torchmetrics installed.')

In [ ]:
# 5c — Verify torch sees the GPU
run_in_env('python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))"')

In [ ]:
# 5d — Install flash-attn via pre-built wheel (auto-detects correct wheel)
import subprocess, sys, re

# Detect actual versions INSIDE the conda env
def query_env(py_expr):
    cmd = f'{CONDA_BIN} run -n avatarforcing --no-capture-output python -c "{py_expr}"'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return r.stdout.strip()

py_ver  = query_env('import sys; print(f"cp{sys.version_info.major}{sys.version_info.minor}")')  # e.g. cp311
torch_v = query_env('import torch; print(torch.__version__.split("+")[0])')                      # e.g. 2.6.0
cuda_v  = query_env('import torch; print(torch.version.cuda)')                                   # e.g. 12.6

torch_short = re.sub(r'(\d+\.\d+)\..*', r'\1', torch_v)      # 2.6.0 -> 2.6
torch_tag   = torch_short.replace('.', '')                      # 2.6   -> 26
cuda_tag    = cuda_v.replace('.', '')                           # 12.6  -> 126

print(f'Detected → Python: {py_ver} | Torch: {torch_v} ({torch_short}) | CUDA: {cuda_v}')
print(f'Tags     → cu{cuda_tag}  torch{torch_tag}  {py_ver}')

# Try cxx11abiFALSE first, then TRUE as fallback
BASE = 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4'
candidates = [
    f'{BASE}/flash_attn-2.7.4+cu{cuda_tag}torch{torch_tag}cxx11abiFALSE-{py_ver}-{py_ver}-linux_x86_64.whl',
    f'{BASE}/flash_attn-2.7.4+cu{cuda_tag}torch{torch_tag}cxx11abiTRUE-{py_ver}-{py_ver}-linux_x86_64.whl',
]

import urllib.request
chosen = None
for url in candidates:
    try:
        req = urllib.request.Request(url, method='HEAD')
        urllib.request.urlopen(req, timeout=10)
        chosen = url
        print(f'✅ Wheel found: {url.split("/")[-1]}')
        break
    except Exception:
        print(f'✗  Not found:  {url.split("/")[-1]}')

if chosen:
    print('\nInstalling (should be < 1 min)...')
    run_in_env(f'pip install "{chosen}" --no-build-isolation -q')
    print('✅ flash-attn installed!')
else:
    print('\n⚠️  No pre-built wheel matched. Check available wheels at:')
    print('   https://github.com/Dao-AILab/flash-attention/releases/tag/v2.7.4')
    print(f'   Look for: cu{cuda_tag}  torch{torch_tag}  {py_ver}  linux_x86_64')
    print('\nFalling back to source build (will take 60-120 min)...')
    run_in_env('pip install flash-attn --no-build-isolation')
    print('✅ flash-attn built from source.')

In [ ]:
# 5e — Final environment verification
run_in_env('python -c "import torch, flash_attn; print(\'torch:\', torch.__version__); print(\'cuda:\', torch.version.cuda); print(\'flash_attn:\', flash_attn.__version__)"')

In [ ]:
print('Installing omegaconf...')
run_in_env(
    'pip install omegaconf'
)
print('✅ omegaconf installed.')

In [ ]:
print('Installing peft...')
run_in_env(
    'pip install peft'
)
print('✅ peft installed.')

In [ ]:
run_in_env('pip install easydict')

In [ ]:
run_in_env('pip install diffusers')

In [ ]:
run_in_env('pip install ftfy')

In [ ]:
run_in_env('pip install lmdb')

In [ ]:
run_in_env('pip install pandas')

In [ ]:
run_in_env('pip install opencv-python')

In [ ]:
run_in_env('pip install decord')

In [ ]:
print('Installing soundfile...')
run_in_env(
    'pip install soundfile'
)
print('✅ soundfile installed.')

In [ ]:
print('Installing system dependency libsndfile...')
run_in_env(
    'conda install -y -c conda-forge libsndfile'
)
print('✅ libsndfile installed.')

In [ ]:
print('Checking torch ecosystem versions...')
run_in_env(
    'python -c "import torch, torchvision, torchaudio; '
    'print(f\'torch: {torch.__version__}\'); '
    'print(f\'torchvision: {torchvision.__version__}\'); '
    'print(f\'torchaudio: {torchaudio.__version__}\')"'
)

In [ ]:
print('Checking installed packages...')
run_in_env(
    'pip list | grep -E "torch|audio|vision"'
)

In [ ]:
wav2vec_path = '/workspace/AvatarForcing-inference-v1/wan/models/wav2vec.py'

with open(wav2vec_path, 'r') as f:
    src = f.read()

# Remove both occurrences of the forbidden line
fixed = src.replace('        self.config.output_attentions = True\n', '')

if fixed == src:
    print('⚠️  Line not found — check file manually')
else:
    with open(wav2vec_path, 'w') as f:
        f.write(fixed)
    count = src.count('        self.config.output_attentions = True')
    print(f'✅ Removed {count} occurrence(s) of self.config.output_attentions = True')
    print('   File saved. Re-run inference now.')

In [ ]:
# 🔧 FIX — Replace torch 2.6.0 with nightly that supports RTX 5090 (sm_120)
print('Uninstalling old torch stack...')
run_in_env('pip uninstall -y torch torchvision torchaudio')

print('Installing PyTorch nightly cu128 (sm_120 / Blackwell support)...')
run_in_env(
    'pip install --pre torch torchvision torchaudio '
    '--index-url https://download.pytorch.org/whl/nightly/cu128'
)
print('✅ Done. Verifying...')
run_in_env(
    'python -c "'
    'import torch; '
    'print(torch.__version__); '
    'print(torch.cuda.get_device_capability()); '
    'x = torch.tensor([1.0]).cuda(); '
    'print(x * 2, \'GPU OK\')'
    '"'
)

In [ ]:
import subprocess
subprocess.run('pkill -9 -f "pip install flash"', shell=True)
subprocess.run('pkill -9 -f "ninja"', shell=True)
subprocess.run('pkill -9 -f "nvcc"', shell=True)
print('✅ Killed.')

In [ ]:
run_in_env('pip uninstall -y flash-attn || true')
print('✅ flash-attn removed.')

In [ ]:
attn_path = '/workspace/AvatarForcing-inference-v1/wan/modules/attention.py'

new_src = '''# Copyright 2024-2025 The Alibaba Wan Team Authors. All rights reserved.
# flash-attn replaced with torch SDPA for RTX 5090 (sm_120) compatibility
import warnings
import torch
import torch.nn.functional as F

FLASH_ATTN_3_AVAILABLE = False
FLASH_ATTN_2_AVAILABLE = False

__all__ = [
    'flash_attention',
    'attention',
]

def flash_attention(
    q, k, v,
    q_lens=None,
    k_lens=None,
    dropout_p=0.,
    softmax_scale=None,
    q_scale=None,
    causal=False,
    window_size=(-1, -1),
    deterministic=False,
    dtype=torch.bfloat16,
    version=None,
):
    """
    Drop-in replacement using PyTorch SDPA (RTX 5090 / sm_120 compatible).
    q: [B, Lq, Nq, C1]
    k: [B, Lk, Nk, C1]
    v: [B, Lk, Nk, C2]
    """
    half_dtypes = (torch.float16, torch.bfloat16)
    assert dtype in half_dtypes
    b, lq, lk, out_dtype = q.size(0), q.size(1), k.size(1), q.dtype

    def half(x):
        return x if x.dtype in half_dtypes else x.to(dtype)

    # Handle variable-length sequences by padding/masking
    if q_lens is not None or k_lens is not None:
        warnings.warn(
            'Padding mask is disabled when using scaled_dot_product_attention. '
            'It can have a significant impact on performance.'
        )

    if q_scale is not None:
        q = q * q_scale

    # q/k/v: [B, L, N, C] -> [B, N, L, C] for SDPA
    q = half(q).transpose(1, 2)
    k = half(k).transpose(1, 2)
    v = half(v).transpose(1, 2)

    scale = softmax_scale if softmax_scale is not None else q.shape[-1] ** -0.5

    out = F.scaled_dot_product_attention(
        q, k, v,
        scale=scale,
        dropout_p=dropout_p if torch.is_grad_enabled() else 0.0,
        is_causal=causal,
    )
    # [B, N, L, C] -> [B, L, N, C]
    return out.transpose(1, 2).to(out_dtype)


def attention(
    q, k, v,
    q_lens=None,
    k_lens=None,
    dropout_p=0.,
    softmax_scale=None,
    q_scale=None,
    causal=False,
    window_size=(-1, -1),
    deterministic=False,
    dtype=torch.bfloat16,
    fa_version=None,
):
    return flash_attention(
        q=q, k=k, v=v,
        q_lens=q_lens, k_lens=k_lens,
        dropout_p=dropout_p,
        softmax_scale=softmax_scale,
        q_scale=q_scale,
        causal=causal,
        window_size=window_size,
        deterministic=deterministic,
        dtype=dtype,
        version=fa_version,
    )
'''

with open(attn_path, 'w') as f:
    f.write(new_src)
print('✅ attention.py fully rewritten — both flash_attention and attention use SDPA.')

In [ ]:
run_in_env('pip install av')

## 📦 Step 6 — Install huggingface_hub & (Optional) Login

In [ ]:
run_in_env("pip install -q 'huggingface_hub[cli]'")

HF_TOKEN = ''  # paste token if needed; all 3 models are public
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    run_in_env(f'huggingface-cli login --token {HF_TOKEN}')
else:
    print('ℹ️  No HF token — public access is fine.')

## ⬇️ Step 7 — Download Models

| Model | Repo | Size |
|---|---|---|
| Wan2.1-T2V-1.3B | `Wan-AI/Wan2.1-T2V-1.3B` | ~5 GB |
| wav2vec2-base-960h | `facebook/wav2vec2-base-960h` | ~360 MB |
| AvatarForcing | `lycui/AvatarForcing` | ~2 GB |

In [ ]:
WAN_DIR     = f'{REPO_DIR}/wan_models/Wan2.1-T2V-1.3B'
WAV2VEC_DIR = f'{REPO_DIR}/wan_models/wav2vec2-base-960h'
CKPT_DIR    = f'{REPO_DIR}/checkpoints'
for d in [WAN_DIR, WAV2VEC_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)
print('Directories ready.')

In [ ]:
if not any(os.scandir(WAN_DIR)):
    run_in_env(f'hf download Wan-AI/Wan2.1-T2V-1.3B --local-dir {WAN_DIR}')
else:
    print(f'✅ Wan model already at {WAN_DIR}')

In [ ]:
if not any(os.scandir(WAV2VEC_DIR)):
    run_in_env(f'hf download facebook/wav2vec2-base-960h --local-dir {WAV2VEC_DIR}')
else:
    print(f'✅ wav2vec2 already at {WAV2VEC_DIR}')

In [ ]:
pt_files = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')]
if not pt_files:
    run_in_env(f'hf download lycui/AvatarForcing --local-dir {CKPT_DIR}')
else:
    print(f'✅ Checkpoints present: {pt_files}')

In [ ]:
for d in [WAN_DIR, WAV2VEC_DIR, CKPT_DIR]:
    print(f'📂 {d}  ({len(os.listdir(d))} files)')

## 🔍 Step 8 — Locate Checkpoint File

In [ ]:
import glob

pt_files = list(set(
    glob.glob(f'{CKPT_DIR}/**/*.pt', recursive=True) +
    glob.glob(f'{CKPT_DIR}/*.pt')
))
for f in pt_files:
    print(f'  {f}  ({os.path.getsize(f)/1e6:.1f} MB)')

preferred = [f for f in pt_files if os.path.basename(f) == 'model.pt']
CHECKPOINT_PATH = preferred[0] if preferred else (pt_files[0] if pt_files else None)
print(f'\n✅ Using: {CHECKPOINT_PATH}')

## 🎬 Step 9 — Batch Configuration (Fixed Inputs + Audio Folder)

**IMAGE_PATH** and **PROMPT** are fixed for every generated video.  
Place all `.wav` / `.mp3` files inside `example/audio/` — the notebook will process them all.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# FIXED INPUTS — identical for every audio in the batch
# ──────────────────────────────────────────────────────────────────────────────
IMAGE_PATH = f'{REPO_DIR}/example/r.png'   # fixed reference portrait

PROMPT = (
    'A close-up, perfectly centered frontal portrait of a bald, feature-minimal '
    'synthetic male face with blue eyes, unique for its block-like cranium and sharp, '
    'angular edges resembling a 3D texture map wrapped over a digital cube. Suspended '
    'against a solid, matching beige background, the floating, earless head remains '
    'completely fixed, rigid, and static in space with zero lateral tilting, turning, '
    'or camera movement. The subject speaks with absolute robotic precision, featuring '
    'high-fidelity lip-syncing where the mouth and jaw movements are sharply articulated '
    'to match phonemes with mechanical accuracy. While the overall blocky head structure '
    'stays locked and temporally consistent, the performance is brought to life by '
    'controlled yet expressive micro-movements, including subtle twitches in the faint '
    'brows, an uncanny eye gaze, and periodic blinking under flat, clinical studio '
    'lighting that highlights detailed skin pores and fine textures without deep shadows.'
)

# ──────────────────────────────────────────────────────────────────────────────
# AUDIO BATCH FOLDER — place .wav / .mp3 files here
# ──────────────────────────────────────────────────────────────────────────────
AUDIO_FOLDER  = f'{REPO_DIR}/example/audio'

# ──────────────────────────────────────────────────────────────────────────────
# OUTPUT FOLDER — videos saved as outputs/<audio_stem>.mp4
# ──────────────────────────────────────────────────────────────────────────────
OUTPUT_FOLDER = f'{REPO_DIR}/outputs'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f'✅ Output folder ready: {OUTPUT_FOLDER}')

# ──────────────────────────────────────────────────────────────────────────────
# INFERENCE CONSTANTS (unchanged from single-inference behaviour)
# ──────────────────────────────────────────────────────────────────────────────
SEED        = 42
NUM_SAMPLES = 1
USE_EMA     = False
VIDEO_FPS   = 25          # pipeline output frame-rate
AUDIO_FPS   = 25          # matches avatarforcing.yaml data.fps
WAV_RATE    = 16000        # Hz — wav2vec fixed sample rate

# ── Validate fixed image ─────────────────────────────────────
assert os.path.isfile(IMAGE_PATH), f'❌ Fixed image not found: {IMAGE_PATH}'
print(f'✅ Fixed image  : {IMAGE_PATH}')
print(f'✅ Prompt length: {len(PROMPT)} chars')

## ⚙️ Step 10 — Scan Audio Folder & Compute Per-File Frame Counts

In [ ]:
import glob, math

SUPPORTED_EXT = {'.wav', '.mp3'}

def get_audio_duration_sec(path):
    """Return audio duration in seconds; supports .wav and .mp3."""
    import soundfile as sf
    try:
        info = sf.info(path)
        return info.duration
    except Exception:
        import torchaudio
        meta = torchaudio.info(path)
        return meta.num_frames / meta.sample_rate

def compute_num_output_frames(duration_sec, video_fps=25):
    """
    Dynamically compute num_output_frames so the video matches audio duration.
    Enforces the pipeline constraint: (num_output_frames - 1) % 4 == 0.
    Minimum 21 frames (~0.8 s).
    """
    raw = math.ceil(duration_sec * video_fps)
    raw = max(raw, 21)                  # never below 21 frames
    remainder = (raw - 1) % 4
    if remainder != 0:
        raw += (4 - remainder)          # round up to next valid count
    return raw

# ── Scan folder ──────────────────────────────────────────────
all_files  = sorted(os.listdir(AUDIO_FOLDER)) if os.path.isdir(AUDIO_FOLDER) else []
audio_jobs = []   # list of (abs_path, stem, num_output_frames, duration_sec)
skipped    = []

if not all_files:
    print(f'⚠️  Audio folder is empty or missing: {AUDIO_FOLDER}')
else:
    for fname in all_files:
        ext  = os.path.splitext(fname)[1].lower()
        full = os.path.join(AUDIO_FOLDER, fname)
        stem = os.path.splitext(fname)[0]

        if ext not in SUPPORTED_EXT:
            print(f'  [SKIP]  {fname}  (unsupported extension "{ext}")')
            skipped.append(fname)
            continue
        if not os.path.isfile(full):
            print(f'  [SKIP]  {fname}  (not a regular file)')
            skipped.append(fname)
            continue
        try:
            dur = get_audio_duration_sec(full)
            if dur < 0.5:
                print(f'  [SKIP]  {fname}  (too short: {dur:.2f}s < 0.5s)')
                skipped.append(fname)
                continue
            nof = compute_num_output_frames(dur, VIDEO_FPS)
            audio_jobs.append((full, stem, nof, dur))
            teacher_len = nof * 4 + 80
            print(
                f'  [OK]    {fname:<30s}  '
                f'dur={dur:.2f}s  frames={nof}  teacher_len={teacher_len}'
            )
        except Exception as e:
            print(f'  [SKIP]  {fname}  (error reading audio: {e})')
            skipped.append(fname)

print(f'\n✅ Audio files to process : {len(audio_jobs)}')
print(f'   Skipped                 : {len(skipped)}')
if not audio_jobs:
    print('⚠️  Nothing to do — add .wav or .mp3 files to the audio folder and re-run this cell.')

## 🚀 Step 11 — Load Model Once, Then Run Batch Inference

The pipeline and wav2vec encoder are loaded **a single time**.  
All audio files are processed inside one loop — no model reloading between files.

In [ ]:
# ============================================================
# CELL 11a — Load Pipeline & Audio Encoder (once)
# ============================================================
import sys, time
import torch
from omegaconf import OmegaConf
from collections import OrderedDict
from torchvision import transforms
from torchvision.io import write_video
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TVF
from einops import rearrange
from PIL import Image
import numpy as np
import subprocess as _sp
import soundfile as sf
import torchaudio

# ── Add repo to Python path ──────────────────────────────────
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from pipeline import AvatarForcingInferencePipeline
from utils.misc import set_seed
from utils.inject import _apply_lora

# ── Config ────────────────────────────────────────────────────
CONFIG_PATH = f'{REPO_DIR}/configs/avatarforcing.yaml'
config      = OmegaConf.load(CONFIG_PATH)
default_cfg = OmegaConf.load(f'{REPO_DIR}/configs/default_config.yaml')
config      = OmegaConf.merge(default_cfg, config)

device = torch.device('cuda')
set_seed(SEED)
torch.set_grad_enabled(False)

# ── Build AvatarForcing pipeline ──────────────────────────────
print('Loading AvatarForcing pipeline ...')
pipeline = AvatarForcingInferencePipeline(config, device=device)

if CHECKPOINT_PATH:
    state_dict = torch.load(CHECKPOINT_PATH, map_location='cpu')
    if USE_EMA:
        sd = state_dict['generator_ema']
        clean = OrderedDict()
        for k, v in sd.items():
            clean[k.replace('_fsdp_wrapped_module.', '')] = v
        sd = clean
    else:
        sd = state_dict['generator']
    pipeline.generator.model = _apply_lora(pipeline.generator.model, config['models']['lora'])
    pipeline.generator.load_state_dict(sd)

pipeline = pipeline.to(device=device, dtype=torch.bfloat16)
print('✅ AvatarForcing pipeline ready.')

# ── Wav2Vec audio encoder ─────────────────────────────────────
from wan.models.wav2vec import Wav2VecModel
from transformers import Wav2Vec2FeatureExtractor

WAV2VEC_DIR = f'{REPO_DIR}/wan_models/wav2vec2-base-960h'
wav_feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC_DIR)
audio_encoder = Wav2VecModel.from_pretrained(WAV2VEC_DIR, local_files_only=True)
audio_encoder.feature_extractor._freeze_parameters()
audio_encoder = audio_encoder.to(device=device, dtype=torch.float32)
audio_encoder.eval()
print('✅ Wav2Vec2 audio encoder ready.')

# ── Image transform (identical to inference.py) ───────────────
class ResizeKeepRatioArea16:
    def __init__(self, area_hw=(480, 832), div=16):
        self.A = area_hw[0] * area_hw[1]
        self.d = div
    def __call__(self, img):
        w, h = img.size
        s = min(1.0, math.sqrt(self.A / (h * w)))
        nh = max(self.d, int(h * s) // self.d * self.d)
        nw = max(self.d, int(w * s) // self.d * self.d)
        return TVF.resize(img, (nh, nw), interpolation=InterpolationMode.BILINEAR, antialias=True)

img_transform = transforms.Compose([
    ResizeKeepRatioArea16((480, 832), 16),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

# ── Pre-encode the FIXED image once ──────────────────────────
print('Encoding fixed reference image ...')
pil_img     = Image.open(IMAGE_PATH).convert('RGB')
img_tensor  = img_transform(pil_img)                                           # [C, H, W]
image       = img_tensor.unsqueeze(0).unsqueeze(2).to(device=device, dtype=torch.bfloat16)  # [1,C,1,H,W]
initial_latent_base = pipeline.vae.encode_to_latent(image).to(device=device, dtype=torch.bfloat16)
print(f'✅ Image encoded — latent shape: {initial_latent_base.shape}')
print()
print('Model load complete. Ready for batch inference.')

In [ ]:
# ============================================================
# CELL 11b — Batch Inference Loop
# Processes every audio file in audio_jobs sequentially.
# Model is NOT reloaded between files.
# ============================================================

def load_and_encode_audio(wav_path, teacher_len, audio_fps=25):
    """
    Reproduce EXACTLY the audio conditioning pipeline from TextImageAudioPairDataset:
      1. Read audio -> resample to 16 kHz mono if needed.
      2. Extract wav2vec input_values via Wav2Vec2FeatureExtractor.
      3. Clip / zero-pad to teacher_len * (16000 / audio_fps) samples.
      4. Run Wav2VecModel with output_hidden_states=True.
      5. Concatenate last_hidden_state + ALL intermediate hidden states.
      6. Prepend one zero prefix frame.
    Returns:
        audio_emb: torch.Tensor  [1, teacher_len+1, hidden_dim]  (batch dim added)
    """
    # --- Read & normalise to 16 kHz mono float32 ---
    audio, sr = sf.read(wav_path, dtype='float32')
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != WAV_RATE:
        audio = torchaudio.functional.resample(
            torch.from_numpy(audio).unsqueeze(0), sr, WAV_RATE
        ).squeeze(0).numpy()

    # --- Wav2Vec feature extraction ---
    input_values = np.squeeze(
        wav_feature_extractor(audio, sampling_rate=WAV_RATE).input_values
    )
    input_values = torch.from_numpy(input_values).float().unsqueeze(0)  # [1, T_samples]

    # --- Clip or pad to exact window (mirrors dataset.py) ---
    max_audio_len = teacher_len * int(WAV_RATE / audio_fps)
    if input_values.shape[1] < max_audio_len:
        pad_len = max_audio_len - input_values.shape[1]
        input_values = torch.cat(
            [input_values, torch.zeros((1, pad_len), dtype=input_values.dtype)], dim=1
        )
    else:
        input_values = input_values[:, :max_audio_len]

    input_values = input_values.to(device=device)

    # --- Encode: replicate exact hidden-state concatenation from dataset.py ---
    with torch.no_grad():
        hs = audio_encoder(input_values, seq_len=teacher_len, output_hidden_states=True)
        audio_emb = hs.last_hidden_state                          # [1, teacher_len, H]
        for h in hs.hidden_states:
            audio_emb = torch.cat([audio_emb, h], dim=-1)        # concat along feature dim

    audio_emb = audio_emb.squeeze(0)                              # [teacher_len, H_total]
    audio_emb = torch.cat(
        [torch.zeros_like(audio_emb[:1]), audio_emb], dim=0      # prepend zero prefix
    )                                                             # [teacher_len+1, H_total]
    return audio_emb.unsqueeze(0)                                 # [1, teacher_len+1, H_total]


# ── Batch counters ────────────────────────────────────────────
successful, failed = [], []
total = len(audio_jobs)

if total == 0:
    print('⚠️  No audio files queued. Add files to the audio folder and re-run Step 10.')
else:
    print('=' * 62)
    print(f'  BATCH START — {total} audio file(s) to process')
    print('=' * 62)
    print()

    for job_idx, (wav_path, stem, num_output_frames, duration_sec) in enumerate(audio_jobs, 1):
        fname       = os.path.basename(wav_path)
        output_path = os.path.join(OUTPUT_FOLDER, f'{stem}.mp4')
        teacher_len = num_output_frames * 4 + 80

        print(f'[{job_idx}/{total}] Processing : {fname}')
        print(f'  Audio Duration : {duration_sec:.2f} sec')
        print(f'  FPS            : {VIDEO_FPS}')
        print(f'  Frames         : {num_output_frames}  (teacher_len={teacher_len})')
        print(f'  Output         : {output_path}')

        t_start = time.time()
        try:
            # ── 1. Encode audio (exact same conditioning pipeline as inference.py) ──
            audio_emb = load_and_encode_audio(wav_path, teacher_len, AUDIO_FPS)
            # audio_emb shape: [1, teacher_len+1, H_total]

            # ── 2. Image latent conditioning (identical to inference.py) ──────────
            initial_latent = initial_latent_base.clone().repeat(NUM_SAMPLES, 1, 1, 1, 1)

            img_lat    = initial_latent.permute(0, 2, 1, 3, 4)
            msk        = torch.zeros_like(
                img_lat.repeat(1, 1, num_output_frames + 20, 1, 1)[:, :1]
            )
            image_cat  = img_lat.repeat(1, 1, num_output_frames + 20, 1, 1)
            msk[:, :, 1:] = 1
            y = torch.cat([image_cat, msk], dim=1)

            h_lat = initial_latent.shape[-2]
            w_lat = initial_latent.shape[-1]

            # ── 3. Sample noise ──────────────────────────────────────────────────
            sampled_noise = torch.randn(
                (NUM_SAMPLES, num_output_frames - 1, 16, h_lat, w_lat),
                device=device,
                dtype=torch.bfloat16,
            )

            # ── 4. Run pipeline.inference_avatar_forcing (exact same call) ───────
            video = pipeline.inference_avatar_forcing(
                noise=sampled_noise,
                text_prompts=[PROMPT] * NUM_SAMPLES,
                audio_embeddings=audio_emb,
                y=y,
                return_latents=False,
                initial_latent=initial_latent,
            )

            # ── 5. Decode frames (identical to inference.py) ─────────────────────
            video = 255.0 * rearrange(video, 'b t c h w -> b t h w c').cpu()

            # ── 6. Clear VAE cache between iterations (VRAM management) ──────────
            pipeline.vae.model.clear_cache()

            # ── 7. Save silent video frames ───────────────────────────────────────
            write_video(output_path, video[0], fps=VIDEO_FPS)

            # ── 8. Mux audio track via ffmpeg (identical to inference.py) ─────────
            tmp_path   = output_path + '.tmp.mp4'
            ffmpeg_cmd = (
                f'ffmpeg -y -i "{output_path}" -i "{wav_path}" '
                f'-c:v copy -c:a aac -shortest "{tmp_path}" '
                f'&& mv "{tmp_path}" "{output_path}"'
            )
            _sp.run(ffmpeg_cmd, shell=True, check=True)

            elapsed = time.time() - t_start
            print(f'  ✅ Inference completed in {elapsed:.1f} sec -> {output_path}')
            print()
            successful.append(fname)

        except Exception as exc:
            elapsed = time.time() - t_start
            import traceback
            print(f'  ❌ ERROR after {elapsed:.1f}s: {exc}')
            traceback.print_exc()
            failed.append(fname)
            # ── Attempt to recover VRAM before continuing ────────────────────
            try:
                pipeline.vae.model.clear_cache()
            except Exception:
                pass
            torch.cuda.empty_cache()
            print(f'  ⚠️  Skipping — continuing with remaining files.')
            print()

    # ── Final summary ─────────────────────────────────────────
    print('=' * 62)
    print('  BATCH COMPLETE')
    print(f'  Successful : {len(successful)}')
    print(f'  Failed     : {len(failed)}')
    if skipped:
        print(f'  Skipped    : {len(skipped)}  {skipped}')
    if failed:
        print(f'  Failed list: {failed}')
    print('=' * 62)
    if successful:
        print(f'\nVideos saved to: {OUTPUT_FOLDER}')
        for f in successful:
            stem = os.path.splitext(f)[0]
            print(f'  {f}  ->  outputs/{stem}.mp4')

## 🎥 Step 12 — Display Output Video

In [ ]:
from IPython.display import Video, display
videos = sorted(glob.glob(f'{OUTPUT_FOLDER}/*.mp4'), key=os.path.getmtime, reverse=True)
if videos:
    print(f'Latest: {videos[0]}  ({os.path.getsize(videos[0])/1e6:.1f} MB)')
    display(Video(videos[0], embed=True, width=832, height=480))
    for v in videos:
        print(f'  {v}')
else:
    print('⚠️  No output videos found.')

## 🩺 Troubleshooting

| Issue | Fix |
|---|---|
| `conda: not found` | Step 1 auto-installs Miniconda |
| `CondaToSNonInteractiveError` | Step 3a accepts TOS |
| `torch==2.5.1+cu121 not found` | Fixed: torch removed from yml; installed via PyTorch index in Step 5 |
| `flash-attn build takes hours` | Fixed: pre-built wheel installed in Step 5d (seconds) |
| `CUDA not available` | Step 5a installs cu126 wheels |
| OOM error | Reduce `NUM_OUTPUT_FRAMES` to 81 |
| `(N-1) % 4 != 0` | Use: 21, 25, 81, 125, 225 |
| No `.pt` checkpoint | Re-run Step 7 download cell |